In [ ]:
%matplotlib inline

In [ ]:
from essential.kegg_modules import list_kegg_modules, kegg_module_to_graph, metabolic_to_operational_graph
from essential.plot_pathways import plot_pathway_results, plot_metabolic_pathway
from essential.utils import PLOTNINE_DEFAULT_THEME_2
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import plotnine as gg
from tqdm import tqdm
import scanpy as sc
from essential.pathway_discontinuity import PathwayDiscontinuity


plt.rcParams["svg.fonttype"] = "none"

def plot_umap_genes(adata, genes):
    obs_subset = adata.obs.loc[lambda x: x["target"].isin(genes)]
    obs_subset["target"] = obs_subset["target"].astype(str)

    fig = (
        gg.ggplot(adata.obs, gg.aes(x="UMAP1", y="UMAP2"))
        + gg.geom_point()
        + gg.geom_point(obs_subset, gg.aes(color="target"), size=2)
        + gg.theme_minimal()
    )
    return fig

def plot_umap_equiv_classes(adata, gene_to_class, class_color_mapping, plot_legend=True, point_size=1.5):
    obs_subset = adata.obs.loc[lambda x: x["target"].isin(gene_to_class.keys())]
    obs_subset["equivalence_class"] = obs_subset["target"].map(gene_to_class).sample(frac=1)

    fig = (
        gg.ggplot(adata.obs, gg.aes(x="UMAP1", y="UMAP2"))
        + gg.geom_point(size=0.7, stroke=0)
        + gg.geom_point(obs_subset, gg.aes(color="equivalence_class"), size=point_size, stroke=0.0)
        + gg.scale_color_manual(values=class_color_mapping)
        + gg.theme_minimal()
    )
    if not plot_legend:
        fig = fig + gg.theme(legend_position="none")
    return fig

#### Import transcriptomic data

In [ ]:
# adata = sc.read_h5ad("/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.h5ad")
adata = sc.read_h5ad("/ewsc/pboyeau/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.h5ad")
# adata.X = adata.layers["reads"]
# adata = adata[~adata.obs["target"].isna()].copy()
# sc.pp.highly_variable_genes(adata, n_top_genes=500, flavor="seurat_v3")
# adata = adata[:, adata.var["highly_variable"]].copy()
# sc.pp.normalize_total(adata)
# sc.pp.log1p(adata)
# sc.pp.pca(adata, n_comps=50)
# For now, just take the PCA from James

sc.pp.neighbors(adata, n_neighbors=10, use_rep="X_pca")
sc.tl.umap(adata, min_dist=0.5)

adata.obs["UMAP1"] = adata.obsm["X_umap"][:, 0]
adata.obs["UMAP2"] = adata.obsm["X_umap"][:, 1]

In [ ]:
mapping = {
   "0": "WT-like baseline",
   "1": "WT-like baseline",
   "2": "WT-like baseline",
   "3": "WT-like baseline",
   "4": "WT-like baseline",
   "5": "Mixed metabolism/respiration",
   "6": "Growth arrest",
   "7": "Envelope stress",
   "8": "Translation",
   "9": "WT-like baseline",
   "10": "Phosphate starvation"
}

adata.obs["leiden_label"] = adata.obs["leiden"].map(mapping)
# sc.pl.umap(adata, color="leiden_label")


import matplotlib as mpl
import plotnine as gg
labels = adata.obs["leiden_label"].astype("category")
tab10 = [mpl.colors.to_hex(c) for c in mpl.cm.get_cmap("tab10").colors]
color_map = {lab: tab10[i % len(tab10)] for i, lab in enumerate(labels.cat.categories)}

fig = (
   gg.ggplot(adata.obs, gg.aes(x="UMAP1", y="UMAP2", color="leiden_label"))
   + gg.geom_point(size=0.5, stroke=0)
   + gg.scale_color_manual(values=color_map)
   + gg.guides(color=gg.guide_legend(override_aes={"size": 3, "alpha": 1}))
   + gg.theme_minimal()
   + PLOTNINE_DEFAULT_THEME_2
   + gg.theme(
      figure_size=(3.5, 2)
   )
   + gg.labs(
      color=""
   )
)
fig.save("figures/leiden_labels.svg")
fig

### scVI experimentsm

In [ ]:
import pandas as pd

obs = adata.obs[["leiden", "target"]].copy()

cluster_sizes = obs.groupby("leiden").size()
total_per_target = obs.groupby("target").size()

def cluster_target_props(grp):
    counts = grp["target"].value_counts()
    props = counts / total_per_target[counts.index]
    return pd.DataFrame({
        "count": counts,
        "proportion": props,
    }).sort_values("count", ascending=False).head(10)

props_by_cluster = obs.groupby("leiden").apply(cluster_target_props)

for cluster in props_by_cluster.index.get_level_values("leiden").unique():
    props = props_by_cluster.loc[cluster]
    print(f"cluster {cluster}")
    print(props)
    print()

In [ ]:
import numpy as np
max_per_cluster = 4000
rng_seed = 0
sampled_index = (
    adata.obs[["leiden"]]
    .groupby("leiden", group_keys=False)
    .apply(lambda g: g.sample(n=min(len(g), max_per_cluster), replace=False, random_state=rng_seed))
    .index.to_numpy()
)
sampled_index

adata_resampled = adata[sampled_index].copy()

In [ ]:
from scvi.model import SCVI

SCVI.setup_anndata(adata_resampled, layer="reads")
model = SCVI(adata_resampled, gene_likelihood="nb")
model.train(
    # batch_size=1024,
    # max_epochs=500,
)

In [ ]:
adata_resampled.obsm["X_scvi_resampled"] = model.get_latent_representation()
sc.pp.neighbors(adata_resampled, use_rep="X_scvi_resampled")
sc.tl.umap(adata_resampled)


In [ ]:
sc.pl.umap(adata_resampled, color="leiden")

In [ ]:
sc.tl.umap(adata, min_dist=0.5)
sc.pl.umap(adata)

#### Mine KEGG modules

In [ ]:
module_info = pd.DataFrame(list_kegg_modules("eco"))
for i, row in tqdm(module_info.iterrows()):
    module_name = row["module_id"]
    g = kegg_module_to_graph(module_name, 'eco')
    genes = np.unique([d['gene'] for u, v, d in g.edges(data=True) if d['gene']])
    np.sort(genes)
    genes_str = ', '.join(genes)

    op = metabolic_to_operational_graph(g)

    pda = PathwayDiscontinuity(adata, representation_obsm_key="X_pca", metabolic_graph=op, perturbation_obs_key="target", global_sigma=5.0)
    results = pda.fit(threshold=0.1, mode="mmd_stat")

    module_info.loc[i, "n_equivalences"] = results.n_equivalences
    module_info.loc[i, "module_has_surprises"] = results.n_equivalences >= 2
    module_info.loc[i, "genes"] = genes_str
    module_info.loc[i, "n_genes"] = len(genes)
module_info.to_csv("module_info.csv", index=False)
module_info.to_json("module_info.json", orient="records", indent=2)


module_prediction = pd.read_json("module_prediction.json")
module_info_ = module_info.merge(module_prediction, on="module_id", how="left").query("n_genes >= 2")

fig = (
    gg.ggplot(
        module_info_,
        gg.aes(x="factor(activity_prediction)", fill="factor(module_has_surprises)")
    ) 
    + gg.geom_bar(position="dodge", width=0.5) 
    + gg.labs(
        x="activity prediction",
        y="# of modules",
        fill="module has surprises"
    ) 
    + gg.theme_minimal()
    + gg.scale_y_continuous(expand=(0, 0))
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(
        figure_size=(2.7, 2)
    )
)
fig.save("figures/module_has_surprises_by_activity_prediction.svg")
fig

In [ ]:
DISPLAY_TOP_K_MODULES = 10

display(module_info_.query("activity_prediction == 'active'").sort_values("n_equivalences", ascending=False).head(DISPLAY_TOP_K_MODULES))

display(module_info_.query("activity_prediction == 'partially active'").sort_values("n_equivalences", ascending=False).head(DISPLAY_TOP_K_MODULES))

display(module_info_.query("activity_prediction == 'inactive'").sort_values("n_equivalences", ascending=False).head(DISPLAY_TOP_K_MODULES))

#### select and visualize a module

In [ ]:
EXAMPLES = [
    "M00938",  # dUTP toxicity,
    "M00120", # "coA biosynthesis"
    "M00063",  # CMP-KDO biosynthesis
    "M00121", # "heme
]

In [ ]:
module_info_.query("activity_prediction == 'active'").query("module_has_surprises").sort_values("n_equivalences", ascending=False)

In [ ]:
# module_name = "eco_M00003"
module_name = "eco_M00060"
g = kegg_module_to_graph(module_name, 'eco', print_info=True)
op = metabolic_to_operational_graph(g)

pda = PathwayDiscontinuity(adata, representation_obsm_key="X_pca", metabolic_graph=op, perturbation_obs_key="target", global_sigma=5.0)
results = pda.fit(threshold=0.1, mode="mmd_stat")
results

In [ ]:
fig, ax, plot_info = plot_pathway_results(
    g, 
    results.edge_equivalence, 
    x_sep=0.5, 
    y_sep=2, 
    label_offset=(0, 0.4), 
    edge_width=1.7, 
    color_surprising_individually=True
)
display(fig)

fig = plot_umap_equiv_classes(adata, plot_info["gene_to_class"], plot_info["class_color_mapping"], plot_legend=False)
display(fig)

In [ ]:
# _ = plot_metabolic_pathway(g, figsize=(5, 5))

In [ ]:
%matplotlib inline

In [ ]:
pairs = results.gene_pair_scores.sort_values("score", ascending=False)
for _, row in pairs.iterrows():
    genes = [row["g1"], row["g2"]]

    fig = plot_umap_genes(adata, genes)
    fig = fig + gg.ggtitle(f"{row['g1']} - {row['g2']}: {row['score']:.2f}")
    display(fig)

# Individual modules

In [ ]:
THRESHOLD = 0.1

## A

In [ ]:
module_id = "eco_M00001"

g = kegg_module_to_graph(module_id, 'eco', print_info=False)
op = metabolic_to_operational_graph(g)

pda = PathwayDiscontinuity(adata, representation_obsm_key="X_pca", metabolic_graph=op, perturbation_obs_key="target", global_sigma=5.0)
results = pda.fit(threshold=THRESHOLD, mode="mmd_stat")

fig, ax, plot_info = plot_pathway_results(
    g, 
    results.edge_equivalence, 
    x_sep=0.5, 
    y_sep=2.5, 
    label_offset=(0, 0.4), 
    edge_width=1.7, 
    color_surprising_individually=True
)
display(fig)
plt.savefig(f"figures/eco_{module_id}.svg")

fig = plot_umap_equiv_classes(adata, plot_info["gene_to_class"], plot_info["class_color_mapping"], plot_legend=False) + PLOTNINE_DEFAULT_THEME_2
fig.save(f"figures/eco_{module_id}_umap.png", dpi=500)
display(fig)


## B

In [ ]:
module_id = "eco_M00121"

g = kegg_module_to_graph(module_id, 'eco', print_info=False)
op = metabolic_to_operational_graph(g)

pda = PathwayDiscontinuity(adata, representation_obsm_key="X_pca", metabolic_graph=op, perturbation_obs_key="target", global_sigma=5.0)
results = pda.fit(threshold=THRESHOLD, mode="mmd_stat")

fig, ax, plot_info = plot_pathway_results(
    g, 
    results.edge_equivalence, 
    x_sep=0.5, 
    y_sep=2.5, 
    label_offset=(0, 0.4), 
    edge_width=1.7, 
    color_surprising_individually=True
)
display(fig)
plt.savefig(f"figures/eco_{module_id}.svg")

fig = plot_umap_equiv_classes(adata, plot_info["gene_to_class"], plot_info["class_color_mapping"], plot_legend=False) + PLOTNINE_DEFAULT_THEME_2
fig.save(f"figures/eco_{module_id}_umap.png", dpi=500)
display(fig)


## C

In [ ]:
module_id = "eco_M00060"

g = kegg_module_to_graph(module_id, 'eco', print_info=False)
op = metabolic_to_operational_graph(g)

pda = PathwayDiscontinuity(adata, representation_obsm_key="X_pca", metabolic_graph=op, perturbation_obs_key="target", global_sigma=5.0)
results = pda.fit(threshold=THRESHOLD, mode="mmd_stat")

fig, ax, plot_info = plot_pathway_results(
    g, 
    results.edge_equivalence, 
    x_sep=0.5, 
    y_sep=2.5, 
    label_offset=(0, 0.4), 
    edge_width=1.7, 
    color_surprising_individually=True
)
display(fig)
plt.savefig(f"figures/eco_{module_id}.svg")

fig = plot_umap_equiv_classes(adata, plot_info["gene_to_class"], plot_info["class_color_mapping"], plot_legend=False) + PLOTNINE_DEFAULT_THEME_2
fig.save(f"figures/eco_{module_id}_umap.png", dpi=500)
display(fig)


# Batched experiment

In [ ]:
selected_modules = module_info_.query("activity_prediction == 'active'").query("module_has_surprises").sort_values("n_equivalences", ascending=False)

In [ ]:
for _, row in selected_modules.iterrows():
    module_id = row["module_id"]
    print(module_id)

    g = kegg_module_to_graph(module_id, 'eco', print_info=False)
    op = metabolic_to_operational_graph(g)

    pda = PathwayDiscontinuity(adata, representation_obsm_key="X_pca", metabolic_graph=op, perturbation_obs_key="target", global_sigma=5.0)
    results = pda.fit(threshold=0.2, mode="mmd_stat")

    fig, ax, plot_info = plot_pathway_results(
        g, 
        results.edge_equivalence, 
        x_sep=0.5, 
        y_sep=2, 
        label_offset=(0, 0.4), 
        edge_width=1.7, 
        color_surprising_individually=True
    )
    # display(fig)
    plt.savefig(f"figures/eco_{module_id}.svg")

    fig = plot_umap_equiv_classes(
        adata, 
        plot_info["gene_to_class"], 
        plot_info["class_color_mapping"], 
        plot_legend=False,
        point_size=5.0,
    )
    # display(fig)
    fig.save(f"figures/eco_{module_id}_umap.png", dpi=500)


In [ ]:
eco_M00001